In [3]:
import sys
import os.path as osp

PROJECT_DIR = '../../'
PROJECT_DIR = osp.abspath(PROJECT_DIR)
print(PROJECT_DIR in sys.path)
if PROJECT_DIR not in sys.path:
    print(f'Adding project directory to the sys.path: {PROJECT_DIR!r}')
    sys.path.insert(1, PROJECT_DIR)

True


Here, we will aim to improve upon the initial two-stage model, which had a good average precision score by recommending the close movies among the top suggestions, but was further away while predicting the values of the ratings themselves. Often this was due to the fact that the model overestimated the ratings by putting all of them in the range between 4 and 5. This result highlights how the candidate generation step with the algorithm based on the Swing algorithm was an effective one, but the ranking step requires further iterational improvement. It should include more information about the items, how much the users closest to the user in question have rated these movies, genres and descriptions of these movies etc. However, we have a pretty limited dataset - apart from the ratings themselves, we have only movie genres, with the rest various user information. With this data availability, it seems reasonable to improve the ranking step based on computing the user-user similarity, then predict movie ratings as a weighted model (not necessarily linear) of the ratings given by these close users. We can even not limit the user number, as this will allow us to calculate the score to almost any movie when the closest user did not watch the closest movie, which can happen in theory.

But this is just one of two main issues to be improved in the initial two-stage model. The other one, which we need to address first, comes from the candidate generation step. Despite being rather effective at selecting the proper candidates closest to the movies already watched by the user in question, the calculations are really slow. It took more than 9 hours to learn the total matrix of the item-to-item similarity scores! While it is true that we have used only a single thread, and the algorithm itself can be optimized to avoid some of the loops, the algorithm is still too slow to apply with the recommendation databases containing thousands of users and items, as it is often in the real-world applications. We just simply need a faster and more efficient way to caclulate the items closest to a selected one.

To combat this, we have turned to the inspirational examples of the real-world systems designed by the leading recommendation companies. Many of them use approximate techniques to compute similarity dur to the enormous resources being required to compute it for all of the items otherwise. One of such techniques is called Locality-Sensitive Hashing, and is widely used for the Approximate Nearest Neighbors (ANN) task. One of its best versions is with the `annoy` library from Spotify, which we will use here. It is a very efficient way to calculate the approximately closest points to the given one, and, as LSH as a whole, is based on building the separating hyperplanes and then assigning the points the binary values based on which side of the hyperplane is each one. The Spotify realization itself also has a lot of the sparse features as does our item data (in form of the users), and so it apparently uses some sort of a very efficient dimensionality reduction algorithm to select around 100 final features which are then passed into LSH. We will emulate this here by using the PCA dimensionality reduction, which will be retrained offline as the new data comes in.

The `annoy` version of LSH itself has the next properties:

- to use `annoy` correctly here, we need to specify the distance as 'euclidean' (it also supports angular distances and several others, which can be very useful sometimes, but here we have a space with the euclidian-measured oldshool distance)
- parameter of the number of trees `n_trees` is used to build a forest of hash functions. This is one of the two tunable parameters in the `annoy` implementation and naturally is extremely important for our performance. The documentation itself onto what it *actually* does was not as straightforward as desired, so I once had to search though the C-written library code to get some insights. But essentially, what it does is:

    1. On the call of the `build()` function this parameter is passed to the program, where it is identified as `q`
    2. The hyperspace (the dimensions of which are set at the start in the initialization of an `AnnoyIndex` object) is split by a randomly generated hyperplane into two sides. The plane is generated by selecting two random points from the current subset, and finding the hyperplane equidistant to them. Then each of these sides (subtrees) is also split by a new hyperplane recursively.
    3. The step 2 is repeated `n_trees` number of times with new randomly generated hyperplanes, forming a forest of `n_trees` trees, or rather random hyperplane splits into buckets.
- So essentially `n_trees` sets a number of random hyperplane buckets split variants that we're getting. The authors recommend to set it, quote, "basically it's recommended to set `n_trees` as large as possible given the amount of memory you can afford." This has an obvious justification, as with the increase of the trees number we get closer and closer to the full Nearest Neighbors solution. Therefore, we'll start from some moderate numbers of trees to optimize the performance, and then we'll possibly increase them to improve the approximation.
- The second parameter, `search_k`, sets a number of nodes searched for a nearest neighbor for an item. It is also recommended to be set as large as possible, with the constraint of the amount of time one can afford to search for the closest neighbors. The `search_k` is set to the default value of `num_neighbors * n_trees`, which is justified by the fact that if we search too few nodes, some of trees end up being underused, and if we search too much, each tree essentially turns into a full-data search. This parameter choice was also proven more than reasonable in practice, as I have used this library before. Therefore, we will most likely stick by it and not tune it any further.
- The parameter `search_k` has a similar story of the implementation as the `n_trees`. However, unlike it, it is not set algorithm-wise, but rather used as a parameter of every `get_nns_by_item()` call. So theoretically it is possible to use different values of it for different items' nearest neighbor search. Then, it sets a number of iterations that the model will spend going through all the trees, starting from their roots. So essentially the default value allows to look at the root of each tree and (n-1) nodes of each one on average.

With the path of development outlined and the algorithms reviewed, let's proceed to realizing them.

In [4]:
from abc import ABC, abstractmethod

In [5]:
!pip install annoy


[notice] A new release of pip is available: 23.3.2 -> 24.2
[notice] To update, run: pip install --upgrade pip


In [7]:
import numpy as np
import pandas as pd
import scipy
from tqdm.notebook import tqdm
import json
import heapq
from datetime import datetime
import concurrent
from more_itertools import grouper
from annoy import AnnoyIndex
from sklearn.decomposition import TruncatedSVD

In [8]:
df_ratings = pd.read_csv('../../data/ml-1m/ratings.dat',
                         delimiter='::',
                         header=None,
                         names=['UserID','MovieID','Rating','Timestamp'],
                         engine ='python')

In [31]:
sparse_ratings_items = scipy.sparse.csr_matrix((df_ratings['Rating'],
                                          (df_ratings['MovieID'] - 1, df_ratings['UserID'] - 1)))
sparse_ratings_items

<3952x6040 sparse matrix of type '<class 'numpy.int64'>'
	with 1000209 stored elements in Compressed Sparse Row format>

In [9]:
from src.models.abstract_rs_model import AbstractRSModel

In [28]:
class TwoStageLSHBasedModel(AbstractRSModel):
    def __init__(self, previous_rows_before_timestamp = 0, n_trees: int = 10, n_components: int = 50):
        self.pre_fit = False
        self.previous_rows_before_timestamp = previous_rows_before_timestamp
        self.previous_date = datetime.fromtimestamp(self.previous_rows_before_timestamp).date()
        self.n_trees = n_trees
        self.n_components = n_components
        # self.lsh = AnnoyIndex(self.n_components, 'angular')
        # self.items_rated_by_user = {}
        # self.users_rated_the_item = {}
        # self.user_item_ratings = {}
        self.sparse_ratings_users = None
        self.sparse_ratings_items = None
        self.pca = None
        self.epsilon = 1e-5
    
    def fit(self, train_data, pre_fit: bool = False):
        if self.pre_fit:
            # The train data was already pre-fit
            pass
        else:
            pass
        self.pre_fit = pre_fit

    def predict(self, data_at_test_timestamp, test_user, test_timestamp):
        # data_at_test_timestamp.shape[0] here is the number of rows
        # before the data for the timestamp in question ends
        if ((datetime.fromtimestamp(test_timestamp).date() - self.previous_date).days >= 7) and (
                data_at_test_timestamp.shape[0] > self.previous_rows_before_timestamp):
            # Here, we are imitating the offline mode where the model updates each week
            # print(f'--Recalculating {datetime.fromtimestamp(test_timestamp).date()} {test_timestamp}')
            self.sparse_ratings_users = scipy.sparse.csr_matrix((df_ratings['Rating'],
                                          (df_ratings['UserID'] - 1, df_ratings['MovieID'] - 1)))
            self.sparse_ratings_items = scipy.sparse.csr_matrix((df_ratings['Rating'],
                                              (df_ratings['MovieID'] - 1, df_ratings['UserID'] - 1)))
            self.pca = TruncatedSVD(n_components=50)
            self.pca.fit(self.sparse_ratings_items)
            # changed_items_list = data_at_test_timestamp[self.previous_rows_before_timestamp:]['MovieID'].unique()
            changed_items_list = data_at_test_timestamp['MovieID'].unique()
            # print(changed_items_list)
            self.lsh = AnnoyIndex(self.n_components, 'angular')
            for changed_item in changed_items_list:
                self.lsh.add_item(changed_item, self.pca.transform(self.sparse_ratings_items[changed_item-1])[0])
            self.lsh.build(self.n_trees)
            self.previous_rows_before_timestamp = data_at_test_timestamp.shape[0]
            self.previous_date = datetime.fromtimestamp(test_timestamp).date()
        # else:
        #     print(f'--Not recalculating {datetime.fromtimestamp(test_timestamp).date()} {test_timestamp}')
        # Candidate generation step
        candidate_items = []
        self.aggregate_scores = {}
        # print(f'--{self.i2i.keys()}')
        # executor = concurrent.futures.ProcessPoolExecutor(8)
        # print(self.sparse_ratings_users[test_user-1].tocoo().col)
        # futures = [executor.submit(self._multiple_already_rated_similarity_scorings, item_already_rated) 
        #            for item_already_rated in self.sparse_ratings_users[test_user-1].tocoo().col]
        # for group in grouper(5, self.sparse_ratings_users[test_user-1].tocoo().col)
        # concurrent.futures.wait(futures)
        for item_already_rated in self.sparse_ratings_users[test_user-1].tocoo().col:
            ids, dists = self.lsh.get_nns_by_vector(
                self.pca.transform(self.sparse_ratings_items[item_already_rated-1])[0],
                20,
                search_k=-1,
                include_distances=True)
            if ids[0] == item_already_rated:
                ids = ids[1:]
                dists = dists[1:]
            # print(item_already_rated, list(zip(ids, dists)))
            if len(dists) > 0:
                dists = 2 - (np.array(dists) - np.min(dists))/(np.max(dists) - np.min(dists) + self.epsilon)
            # if ids[0] == 0:
            #     print(ids, dists)
            for i_c, item_compared in enumerate(ids):
                self.aggregate_scores[item_compared] = self.aggregate_scores.get(item_compared, 0.0) + (20 - i_c)*dists[i_c]
            # if self.sparse_ratings_users[test_user-1,item_already_rated-1] >= 4:
            #     candidate_items += ids[:5]
        candidate_items += heapq.nsmallest(50,
                                               self.aggregate_scores,
                                               key=self.aggregate_scores.get)
        # print(changed_items_list, candidate_items)
        candidate_items = list(np.unique(candidate_items))
        # Ranking step
        aggregate_ratings = {}
        for candidate_item in candidate_items:
            if self.sparse_ratings_items[candidate_item-1].getnnz() > 0:
                # if self.aggregate_scores[candidate_item] == 0:
                # print(candidate_item, self.aggregate_scores[candidate_item], self.sparse_ratings_items[candidate_item-1].tocoo().data)
                aggregate_ratings[candidate_item] = self.aggregate_scores[candidate_item]*(
                    (self.sparse_ratings_items[candidate_item-1].tocoo().data.mean() - 1)/4 + 1)
        ranked_candidates = np.array(sorted(aggregate_ratings, key=aggregate_ratings.get, reverse=True))
        ranked_candidates_scores = np.array(sorted(aggregate_ratings.values(), reverse=True))
        if len(ranked_candidates_scores) > 0:
            # print(ranked_candidates_scores.max(),
            #       ranked_candidates_scores.min(),
            #       ranked_candidates,
            #       ranked_candidates_scores)
            ranked_candidates_scores = (ranked_candidates_scores - ranked_candidates_scores.min())/(
                ranked_candidates_scores.max() - ranked_candidates_scores.min()) + 4
        return ranked_candidates, ranked_candidates_scores

    def fit_predict(self, data, test_user, test_timestamp):
        self.fit(data)
        return self.predict(data, test_user, test_timestamp)

    def _multiple_already_rated_similarity_scorings(self, items):
        for item_already_rated in items:
            ids, dists = self.lsh.get_nns_by_vector(
                self.pca.transform(self.sparse_ratings_items[item_already_rated-1])[0],
                20,
                search_k=-1,
                include_distances=True)
            if ids[0] == item_already_rated:
                ids = ids[1:]
                dists = dists[1:]
            # print(item_already_rated, list(zip(ids, dists)))
            dists = np.array(dists)/dists[0]
            # if ids[0] == 0:
            #     print(ids, dists)
            for i_c, item_compared in enumerate(ids):
                self.aggregate_scores[item_compared] = self.aggregate_scores.get(item_compared, 0.0) + (20 - i_c)*dists[i_c]
            # if self.sparse_ratings_users[test_user-1,item_already_rated-1] >= 4:
            #     candidate_items += ids[:5]

In [55]:
df_ratings['Timestamp'].unique()[-10000]

957459514

In [32]:
%%time
items_pred, ratings_pred = TwoStageLSHBasedModel(n_trees=40, n_components=50).fit_predict(
    df_ratings[df_ratings['Timestamp'] < 957459514], 6035, 957459514) # 1028

CPU times: user 8.87 s, sys: 6.26 s, total: 15.1 s
Wall time: 2.7 s


In [77]:
print(list(zip(items_pred[:20], ratings_pred[:20])))

[(2974, 5.0), (452, 5.070912627047899), (1303, 5.075117068344513), (2847, 5.0893689542541205), (3520, 5.092105635281989), (1177, 5.145220894096655), (2950, 5.356614542160561), (2819, 5.38818302378815), (1006, 5.400520975839176), (2940, 5.404283666826378), (2135, 5.414540296250209), (1979, 5.421472072501896), (2413, 5.4271956437785125), (2070, 5.440427880638446), (3263, 5.446984539107644), (3146, 5.449926041658124), (861, 5.453317944590836), (2692, 5.466358693687714), (265, 5.4825835641708505), (2430, 5.497222779301415)]


In [34]:
print(list(zip(items_pred[:20], ratings_pred[:20])))

[(1299, 5.0), (1089, 4.981093071996539), (3200, 4.9499585935031165), (3342, 4.938719385732981), (1285, 4.934891919793224), (2618, 4.9333411432889225), (2511, 4.930083327902949), (3415, 4.929941704966842), (235, 4.910861437409951), (2109, 4.904070013384376), (934, 4.8983087915020365), (1042, 4.86605995466985), (1009, 4.803883012286259), (170, 4.7951745958377625), (2297, 4.785647990720541), (2793, 4.721124263636853), (2259, 4.714127607453995), (1254, 4.5503749240810905), (2064, 4.5235079916380565), (2183, 4.50587062394771)]


And now, let's evaluate our developed two-stage LSH-based Approximate Nearest Neighbors model in our evaluation framework:

In [22]:
from src.evaluation import EvaluationPipeline

In [35]:
eval_two_stage_lsh = EvaluationPipeline(df_ratings.sample(frac=0.1, random_state=5), 0.2)

In [36]:
metrics_output_dict_two_stage_lsh = eval_two_stage_lsh.evaluate( # recommendation_results_baseline
    TwoStageLSHBasedModel(),
    user_average_metrics=False)

  0%|          | 0/19957 [00:00<?, ?it/s]

In [37]:
metrics_output_dict_two_stage_lsh

{'mae': 3.428929985503974,
 'rmse': 3.6158218389168257,
 'precision': 0.00030064638973793654,
 'average_precision': 0.6383051252324606,
 'mean_reciprocal_rank': 0.0015537653322355688,
 'ndcg': 0.943313403817395,
 'coverage': 0.20433540581414888}

In [38]:
with open('two_stage_lsh_based_metrics.json', 'w') as f:
    json.dump(metrics_output_dict_two_stage_lsh, f)

There are several key moments to notice here:

- first, the computational time has now significantly decreased compared to the initial two-stage model. Moreover, it does not rise significantly if we switch for weekly to daily, or even hourly offline model updates. The LSH-based ANN procedure, including the dimensionality reduction step, is so optimal by itself that it takes time only compared to less than 10 usual predictions. It can be optimized even further with a careful tweaking of `annoy`, as it does not permit enlarging the item set after it was built. With transforming the data to other structure it can be possible to not continuuously re-feed LSH the new items when they are first rated.
- Still, the whole evaluating process took a total of just over 1 hour and 23 minutes. Mainly, this is attributed to bthe two for-loops over already rated items and candidates on respectfully the candidate generation and ranking steps. Therefore, some multiprocessing and vectorization should be looked into more deeple. LSH usually presumes very optimal  data structures feeding and retrieving the information from it. Then it truly shines, allowing the enormous recommender engines such as Spotify to make recommendations in minimum computational steps.
- The model is very good at finding the items that the user should watch in the future, as is the initial model. This results in the average precision being 0.638, and improving upon its value in the previous model. It looks like item-to-item models are really good at finding movies that were watched by a close subgroup of people. Worth noting, that in out evaluation pipeline avberage precision is capped at the number of the ratings each user still has in test. Therefore, it is calculated for a different number of points for each user, which, according to our chosen train-test split proportion, is equal to the 20% of the user's overall ratings. Therefore, the model has reached such mean average precision over the 20% of the latest ratings made by each user.
- At the same time, the precision itself (so the top-1 precision, generalizing it) is still close to 0. Some reasons for this were described for the previous models, main from which is the fact that we have no way to rank the ground truth user's ratings of 5, and therefore know the top pick. Therefore, the precision value for our dataset is largely useless.
- `MAE` and `RMSE` show how this model was still significantly overestimating the ratings given by the user in the future. Once again, we are stuck with trying to convert our scores to the ratings distribution without fitting them to the explicit model. But more important aspect here is that our model is able to largely detect the movies potentially interesting to the user (as evidenced by the average precision), and to approximately rank them in the right order (as shown by the NDCG reaching 0.943 here).
- At the same time, while the mean reciprocal rank value is 3 times higher than for the initial two-stage model, its still low value shows that while the relevant movies do appear on the recommendation, they do not do it straight away. This confirms the strong point of this model - the LSH-based candidate generation step which is able to select the items; and and the weaker point - the ranking step. In the ranking step, we for the sake of speed have decided to not use any complicated ranking schemas, given that the ground truth ranking itself is kind of ambiguous due to low user rating discretization. So, even focusing on including the user-given ratings of the respective movies is not completely useful for this task. Ranking step most likely needs another offline similarity calculation step, whis time with the user-user similarity. Then. the predicted ratings of each movies will be the actual ratings and not approximated scores, as it is here.
- And finally, look at the coverage! Over 20% of the items were in the top recommendations, which is a big increase from even the previous model. This can be partially attributed to the random factor in the LSH algorith, which allows spreading out the highest predictions. The other contributing factor in this is the ranking itself, which takes into account mean ratings together with the item positioning in the previously computed similarity.

### Pros and cons of the two-stage approach

To the main advantages of such an approach in general should be attributed:
- Ability to quickly filter through the full item space, not wasting resources estimating ranking or complicated scores for the items that will never be realistically recommended. Instead, we have an opportunity to quickly choose all potentially useful items, and start proper score computations and ordering only for them.
- All of the features or models on the ranking step can be dedicated to just finding the elements with the best fit for the user, with the candidate generation step making sure that all of them fit at least marginally in some way. Therefore, the model can produce the rankings directly, not taking into account that all the relevant items should be ranked higher than irrelevant. The goal of the first stage therefore can be described as trying to maximize recall while keeping the number of candidates reasonable, while the second stage can focus more on precision of the recommendations. But the first stage can improve precision too, as we have aimed with this model in this work.
- By essentially having two systems in place, we can focus on the different domain on each step. The classical example would be the candidate geberation step focusing on the item-to-item and item-to-user dimensions, while the ranking step will take user-to-user properties into consideration. And while the model with no specific candidate/ranking steps can also use all of them, the two-stage approach offers a specialization to each step.
- This allows us to essentially dedicate the first stage to preprocessing - building similarity matrices, embeddings, filtering the data etc. At the same time, the second stage can focus more on the Machine Learning and model development part of the work.
- By using both the item and user data, this approach allows fast decisions in the case where either of these dimensions does not have any data (or the data is limited). Therefore, it significantly eases the handing of the new items and users.

The main disadvantages include:
- The most obvious drawback is that if the item that can be relevant for the user is not selected on the first stage it is not able to recover on the second stage at all. A good example of this can be an item that is not really similar to the ones the user has already watched, but is very popular among the alike users. The system relying heavily on the item-to-item exploration on the first stage will just miss it despite its high ratings with the similar users.
- The candidate generation step may require the ranking on its own to choose the candidate, therefore creating a situation where the computational gains from the approach are minimal.
- The models can be more complex and difficult to interpret across all the approaches.
- As the real-world applications most likely will require approximate solutions to decrease the computation time, the resulting errors in them can layer over each other for the certain point and give unrelevant recommendations.